# NRP-Exp: Reproducible SoftwareX Example

This notebook reproduces the programmatic workflow described in the
NRP-Exp SoftwareX article.

The example demonstrates:

1. Loading an NRP instance from JSON.
2. Aggregating multivalued evaluator attributes.
3. Executing the ordered preprocessing pipeline.
4. Formulating a multi-objective optimization problem.
5. Generating a Pareto front.
6. Computing Pareto-front quality metrics.
7. Enriching the solutions with NRP indicators.
8. Applying post-optimization analytical lenses.

The notebook uses only the importable Python engine. It does not import
Streamlit or any module from `src.interface`.

Software version: `v1.0.0`

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """
    Locate the project root by searching for the src directory.
    """
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing the 'src' directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\imagu\OneDrive\Escritorio\NRPExperiment_rename


In [2]:
import json
import platform
import time

import numpy as np
import pandas as pd

from src.domain.core.loader import GenericJSONLoader
from src.domain.core.pipeline import PreprocessingEngine
from src.domain.plugins.nrp import NRPPlugin
from src.solving.algorithms.greedy import GreedySolver
from src.solving.core.metrics import ParetoMetrics
from src.solving.core.problem import (
    ConstraintSpec,
    OptimizationProblem,
    ThresholdOperator,
)

from src.analysis.lenses.diversity import DiversityLens
from src.analysis.lenses.preference import PreferenceLens

In [4]:
import scipy
import sklearn


environment = {
    "Python": platform.python_version(),
    "Platform": platform.platform(),
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
}

pd.Series(environment, name="Version").to_frame()

,Version
Python,3.13.14
Platform,Windows-11-10.0.26200-SP0
NumPy,2.5.2
Pandas,3.0.5
SciPy,1.18.0
scikit-learn,1.9.0


## 1. Load the NRP instance

The example uses the `Concise_Case_11` instance distributed with the
repository. The instance contains evaluator groups, multivalued attributes,
precedences, couplings, exclusions, and value dependencies.

Multivalued attributes are aggregated during loading. The preprocessing
engine subsequently applies coupling, value-dependency, and exclusion
processing in that order.

In [7]:
INSTANCE_PATH = (
    PROJECT_ROOT
    / "examples"
    / "instances"
    / "Concise_Case_11.json"
)

if not INSTANCE_PATH.exists():
    raise FileNotFoundError(
        f"Example instance not found: {INSTANCE_PATH}"
    )

raw_json_text = INSTANCE_PATH.read_text(encoding="utf-8")
raw_json_data = json.loads(raw_json_text)

print(f"Instance: {raw_json_data.get('name', INSTANCE_PATH.stem)}")
print(f"File: {INSTANCE_PATH.relative_to(PROJECT_ROOT)}")

Instance: Concise_Case_11
File: examples\instances\Concise_Case_11.json


In [8]:
raw_requirements = (
    raw_json_data.get("requirements")
    or raw_json_data.get("items")
    or []
)

raw_relationships = raw_json_data.get("relationships", {})

raw_summary = {
    "Requirements": len(raw_requirements),
    "Stakeholders": len(raw_json_data.get("stakeholders", [])),
    "Developers": len(raw_json_data.get("developers", [])),
    "Precedences": len(
        raw_relationships.get("precedences", [])
        or raw_relationships.get("precedence", [])
    ),
    "Couplings": len(
        raw_relationships.get("couplings", [])
        or raw_relationships.get("coupling", [])
    ),
    "Exclusions": len(
        raw_relationships.get("exclusions", [])
        or raw_relationships.get("exclusion", [])
    ),
    "Value dependencies": len(
        raw_relationships.get("value_dependencies", [])
        or raw_json_data.get("value_dependencies", [])
    ),
}

pd.Series(raw_summary, name="Original instance").to_frame()

,Original instance
Requirements,12
Stakeholders,2
Developers,2
Precedences,11
Couplings,1
Exclusions,1
Value dependencies,3


Load a model and multivalued aggregation

In [12]:
base_model = GenericJSONLoader.load(
    json_input=raw_json_text,
    plugin=NRPPlugin,
)

print(f"Loaded model: {base_model.name}")
print(f"Items after loading: {len(base_model.items)}")
print(
    "Available attributes:",
    ", ".join(base_model.get_available_attributes()),
)

Loaded model: Concise_Case_11
Items after loading: 12
Available attributes: dissatisfaction, effort, risk, satisfaction


Check multivalued dependencies

In [13]:
aggregation_examples = []

for raw_requirement in raw_requirements:
    requirement_id = str(raw_requirement.get("id", "")).strip()
    raw_attributes = raw_requirement.get("attributes", {})

    if requirement_id not in base_model.items:
        continue

    for attribute_name, raw_value in raw_attributes.items():
        if isinstance(raw_value, (list, dict)):
            clean_name = NRPPlugin.clean_attribute_name(attribute_name)

            aggregated_value = base_model.items[
                requirement_id
            ].attributes.get(clean_name)

            aggregation_examples.append(
                {
                    "requirement": requirement_id,
                    "attribute": clean_name,
                    "original_value": raw_value,
                    "aggregated_value": aggregated_value,
                }
            )

aggregation_df = pd.DataFrame(aggregation_examples)

if aggregation_df.empty:
    print("The example does not contain multivalued attributes.")
else:
    display(aggregation_df.head(10))

,requirement,attribute,original_value,aggregated_value
0,r1,risk,"[2, 3]",2.333333
1,r1,satisfaction,"[1, 5]",3.857143
2,r1,dissatisfaction,"[0, 1]",0.714286
3,r2,risk,"[3, 3]",3.000000
4,r2,satisfaction,"[1, 1]",1.000000
5,r2,dissatisfaction,"[1, 2]",1.714286
6,r3,risk,"[1, 2]",1.333333
7,r3,satisfaction,"[2, 1]",1.285714
8,r3,dissatisfaction,"[1, 3]",2.428571
9,r4,risk,"[2, 2]",2.000000


Verification

In [14]:
for item in base_model.items.values():
    for attribute_name, attribute_value in item.attributes.items():
        assert not isinstance(
            attribute_value,
            (list, tuple, dict),
        ), (
            f"Attribute '{attribute_name}' of item '{item.id}' "
            "was not aggregated before preprocessing."
        )

print("All multivalued attributes were aggregated successfully.")

All multivalued attributes were aggregated successfully.


## 2. Execute the NRP preprocessing pipeline

The NRP preprocessing order is:

1. Multivalued aggregation during JSON loading.
2. Coupling resolution and item fusion.
3. Value-dependency adjustment.
4. Exclusion branching.

Coupling remaps precedences, exclusions, and value dependencies to the
identifiers of the unified items. Value dependencies internal to a unified
item are absorbed into its attributes. Dependencies that continue to span
different items remain available for solution-level evaluation.

In [15]:
processed_models = PreprocessingEngine.process(
    initial_model=base_model,
    pipeline_options={},
)

if not processed_models:
    raise RuntimeError(
        "The preprocessing pipeline did not produce any model."
    )

print(f"Input items: {len(base_model.items)}")
print(f"Generated variants: {len(processed_models)}")

variant_summary = pd.DataFrame(
    [
        {
            "variant_index": index,
            "branch_name": getattr(
                model,
                "branch_name",
                f"Variant {index + 1}",
            ),
            "items": len(model.items),
            "precedences": len(
                model.relationships.get("precedences", [])
            ),
            "remaining_couplings": len(
                model.relationships.get("couplings", [])
            ),
            "exclusions": len(
                model.relationships.get("exclusions", [])
            ),
            "dynamic_value_dependencies": len(
                model.relationships.get(
                    "value_dependencies",
                    [],
                )
            ),
        }
        for index, model in enumerate(processed_models)
    ]
)

variant_summary

Input items: 12
Generated variants: 2


,variant_index,branch_name,items,precedences,remaining_couplings,exclusions,dynamic_value_dependencies
0,0,Excl(r7),10,9,0,0,3
1,1,Excl(r9),8,7,0,0,3


Metadata verification

In [16]:
for index, processed_model in enumerate(processed_models):
    assert (
        processed_model.attribute_definitions
        == base_model.attribute_definitions
    ), (
        f"Variant {index} did not preserve "
        "attribute_definitions."
    )

    assert (
        processed_model.metadata
        == base_model.metadata
    ), f"Variant {index} did not preserve metadata."

    assert not processed_model.relationships.get(
        "couplings",
        [],
    ), f"Variant {index} contains unresolved couplings."

print(
    "All variants preserve attribute definitions and metadata, "
    "and all coupling relations were resolved."
)

All variants preserve attribute definitions and metadata, and all coupling relations were resolved.


Variants review.

In [17]:
SELECTED_VARIANT_INDEX = 0

if SELECTED_VARIANT_INDEX >= len(processed_models):
    raise IndexError(
        f"Variant {SELECTED_VARIANT_INDEX} is not available. "
        f"The pipeline produced {len(processed_models)} variants."
    )

model = processed_models[SELECTED_VARIANT_INDEX]

print(
    "Selected variant:",
    getattr(
        model,
        "branch_name",
        f"Variant {SELECTED_VARIANT_INDEX + 1}",
    ),
)

print(f"Active items: {len(model.items)}")

Selected variant: Excl(r7)
Active items: 10


Inspection of combined requirements for coupling

In [19]:
merged_items = [
    item_id
    for item_id in model.items
    if "_" in str(item_id)
]

print("Unified items:", merged_items or "None")

if merged_items:
    merged_item_table = pd.DataFrame(
        [
            {
                "item_id": item_id,
                "description": model.items[item_id].description,
                **model.items[item_id].attributes,
            }
            for item_id in merged_items
        ]
    )

    

Unified items: ['r4_r5']


## 3. Formulate the multi-objective problem

The illustrative run maximizes stakeholder satisfaction and minimizes
development effort. A maximum-effort constraint is imposed to exclude
release plans that exceed the available capacity.

The exact objective names and capacity value used below are part of the
reproducibility configuration and must remain unchanged for reproducing
the published results.


In [20]:
OBJECTIVES = {
    "satisfaction": "max",
    "effort": "min",
}

available_attributes = set(
    model.get_available_attributes()
)

missing_objectives = set(OBJECTIVES) - available_attributes

if missing_objectives:
    raise ValueError(
        "The selected objectives are not available in the model: "
        f"{sorted(missing_objectives)}"
    )

print("Configured objectives:")
for objective_name, direction in OBJECTIVES.items():
    print(f"  {objective_name}: {direction}")

Configured objectives:
  satisfaction: max
  effort: min


Configuration: Bounds

In [21]:
EFFORT_LIMIT = 100.0

constraints = {
    "effort": ConstraintSpec(
        attribute="effort",
        operator=ThresholdOperator.LESS_EQUAL,
        value=EFFORT_LIMIT,
    )
}

problem = OptimizationProblem(
    model=model,
    objectives=OBJECTIVES,
    constraints=constraints,
)

print("Problem configuration")
print(f"  Model: {model.name}")
print(f"  Variant index: {SELECTED_VARIANT_INDEX}")
print(f"  Objectives: {OBJECTIVES}")
print(f"  Maximum effort: {EFFORT_LIMIT}")

Problem configuration
  Model: Concise_Case_11
  Variant index: 0
  Objectives: {'satisfaction': 'max', 'effort': 'min'}
  Maximum effort: 100.0


## 4. Generate a Pareto front

This example uses the Greedy Multi-Weight Sweep solver to demonstrate
the common programmatic solver interface. The solver constructs solutions
for several objective-weight combinations and returns the non-dominated
solutions found during the sweep.

The result should be interpreted as an approximation produced by the
configured weight sweep, not as a guarantee that every Pareto-optimal

In [22]:
SOLVER_PARAMETERS = {
    "weight_samples": 25,
}

solver = GreedySolver(**SOLVER_PARAMETERS)

start_time = time.perf_counter()
pareto_front = solver.solve(problem)
execution_time = time.perf_counter() - start_time

pareto_front.set_objective_directions(
    problem.get_objective_directions()
)

pareto_df = pareto_front.to_dataframe()

print(f"Solver: {solver.__class__.__name__}")
print(f"Parameters: {SOLVER_PARAMETERS}")
print(f"Execution time: {execution_time:.6f} seconds")
print(f"Pareto-front size: {len(pareto_front)}")

Solver: GreedySolver
Parameters: {'weight_samples': 25}
Execution time: 0.104216 seconds
Pareto-front size: 25


Front Verification

In [23]:
assert len(pareto_front) > 0, (
    "The solver returned an empty Pareto front."
)

assert not pareto_df.empty, (
    "The Pareto-front DataFrame is empty."
)

for objective_name in OBJECTIVES:
    assert objective_name in pareto_df.columns, (
        f"Objective column '{objective_name}' is missing."
    )

assert all(
    solution.is_feasible
    for solution in pareto_front.solutions
), "The Pareto front contains infeasible solutions."

print("The Pareto front is non-empty and all solutions are feasible.")


The Pareto front is non-empty and all solutions are feasible.


Inspect the Front

In [24]:
display_columns = [
    column
    for column in [
        "id",
        "satisfaction",
        "effort",
        "selected_ids",
        "selected_ids_str",
    ]
    if column in pareto_df.columns
]

pareto_df[display_columns].head(20)

,id,satisfaction,effort,selected_ids,selected_ids_str
0,G_1,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
1,G_2,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
2,G_3,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
3,G_4,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
4,G_5,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
5,G_6,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
6,G_7,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
7,G_8,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
8,G_9,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
9,G_10,19.428571,42.0,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"


Front Metrics 

In [25]:
pareto_metrics = ParetoMetrics.compute_all_metrics(
    pareto_front,
    objective_names=list(OBJECTIVES.keys()),
)

metrics_record = {
    "Solver": solver.__class__.__name__,
    "Variant": SELECTED_VARIANT_INDEX,
    "Execution time (s)": execution_time,
    "Weight samples": SOLVER_PARAMETERS["weight_samples"],
    **pareto_metrics,
}

pd.Series(
    metrics_record,
    name="Illustrative execution",
).to_frame()

,Illustrative execution
Solver,GreedySolver
Variant,0
Execution time (s),0.104216
Weight samples,25
Solutions,25
Hypervolume,1.0
Spacing,0.0
Spread,0.0


Export

In [26]:
RESULTS_DIRECTORY = PROJECT_ROOT / "examples" / "results"
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

PARETO_OUTPUT_PATH = (
    RESULTS_DIRECTORY
    / "greedy_pareto.csv"
)

SUMMARY_OUTPUT_PATH = (
    RESULTS_DIRECTORY
    / "execution_summary.csv"
)

pareto_df.to_csv(
    PARETO_OUTPUT_PATH,
    index=False,
)

pd.DataFrame([metrics_record]).to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

print(
    "Pareto front saved to:",
    PARETO_OUTPUT_PATH.relative_to(PROJECT_ROOT),
)

print(
    "Execution summary saved to:",
    SUMMARY_OUTPUT_PATH.relative_to(PROJECT_ROOT),
)

Pareto front saved to: examples\results\greedy_pareto.csv
Execution summary saved to: examples\results\execution_summary.csv


## 5. Semantic enrichment

NRP-Exp can derive domain-specific indicators from the objective values
and requirement selections. The exact indicators available depend on the
attributes contained in the instance and the columns present in the
Pareto-front DataFrame.

In [27]:
indicator_definitions = (
    NRPPlugin.get_calculated_indicators()
)

print("Indicators registered by the NRP plugin:")

for indicator_name, definition in indicator_definitions.items():
    description = getattr(
        definition,
        "description",
        "",
    )

    print(f"  {indicator_name}: {description}")

Indicators registered by the NRP plugin:
  scope: Ratio of included requirements.
  productivity: Satisfaction per unit of effort.
  effectiveness: Satisfaction per unit of financial cost.
  squandering: Wasted capacity relative to maximum effort.
  dirtiness: Dissatisfaction per unit of effort.
  annoyance: Dissatisfaction relative to satisfaction.
  stickiness: Prevalence per unit of effort.
  robustness: Satisfaction versus instability.
  fragility: Failure risk due to prevalence and instability.
  response: Technical response speed (effort per time).
  opportunity: Time usage versus satisfaction.
  usage_efficiency: Prevalence per unit of financial cost.


Indicators calculation

In [28]:
REQUESTED_INDICATORS = [
    "scope",
    "productivity",
    "effectiveness",
]

available_indicator_names = set(
    indicator_definitions.keys()
)

selected_indicators = [
    indicator
    for indicator in REQUESTED_INDICATORS
    if indicator in available_indicator_names
]

if not selected_indicators:
    raise ValueError(
        "None of the requested enrichment indicators "
        "is registered by the NRP plugin."
    )

enriched_df = NRPPlugin.compute_indicators(
    pareto_df.copy(),
    selected_indicators,
    decision_var_prefix="req_",
)

new_indicator_columns = [
    column
    for column in selected_indicators
    if column in enriched_df.columns
]

print("Requested indicators:", selected_indicators)
print("Computed indicator columns:", new_indicator_columns)

[NRPEnricher] Could not compute indicator 'effectiveness': 'cost'


Requested indicators: ['scope', 'productivity', 'effectiveness']
Computed indicator columns: ['productivity']


Full data inspection

In [29]:
enriched_display_columns = [
    column
    for column in [
        "id",
        "satisfaction",
        "effort",
        *new_indicator_columns,
    ]
    if column in enriched_df.columns
]

enriched_df[enriched_display_columns].head(20)

,id,satisfaction,effort,productivity
0,G_1,19.428571,42.0,0.462585
1,G_2,19.428571,42.0,0.462585
2,G_3,19.428571,42.0,0.462585
3,G_4,19.428571,42.0,0.462585
4,G_5,19.428571,42.0,0.462585
5,G_6,19.428571,42.0,0.462585
6,G_7,19.428571,42.0,0.462585
7,G_8,19.428571,42.0,0.462585
8,G_9,19.428571,42.0,0.462585
9,G_10,19.428571,42.0,0.462585


## 6. Post-optimization analytical lenses

The following cells demonstrate two complementary readings of the same
Pareto front:

- Diversity analysis groups solutions according to their position in the
  selected dimensions.
- Preference analysis ranks solutions using explicit optimization
  directions.

These outputs are treated as complementary Sets of Interest rather than
as interchangeable estimates of one universally correct ranking.

In [30]:
excluded_columns = {
    "id",
    "selected_ids",
    "selected_ids_str",
    "is_feasible",
}

numeric_dimensions = [
    column
    for column in enriched_df.select_dtypes(
        include="number"
    ).columns
    if (
        column not in excluded_columns
        and not str(column).startswith("req_")
    )
]

preferred_diversity_dimensions = [
    column
    for column in [
        "satisfaction",
        "effort",
        "scope",
        "productivity",
    ]
    if column in numeric_dimensions
]

if len(preferred_diversity_dimensions) < 2:
    preferred_diversity_dimensions = (
        numeric_dimensions[:4]
    )

if len(preferred_diversity_dimensions) < 2:
    raise ValueError(
        "At least two numeric dimensions are required "
        "for diversity analysis."
    )

n_solutions = len(enriched_df)
diversity_k = min(
    3,
    max(1, n_solutions - 1),
)

diversity_params = {
    "method": "K-Means",
    "cluster_metrics": preferred_diversity_dimensions,
    "k": diversity_k,
}

diversity_lens = DiversityLens()

diversity_df = diversity_lens.apply(
    enriched_df.copy(),
    diversity_params,
)

print("Diversity dimensions:", preferred_diversity_dimensions)
print("Configured clusters:", diversity_k)

Diversity dimensions: ['satisfaction', 'effort', 'productivity']
Configured clusters: 3


c:\Users\imagu\OneDrive\Escritorio\NRPExperiment_rename\venv\Lib\site-packages\sklearn\base.py:1403: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Results

In [31]:
diversity_columns = [
    column
    for column in [
        "id",
        *preferred_diversity_dimensions,
        "cluster",
        "cluster_str",
        "cluster_size",
        "diversity_method",
    ]
    if column in diversity_df.columns
]

diversity_df[diversity_columns].head(20)

,id,satisfaction,effort,productivity,cluster,cluster_str,diversity_method
0,G_1,19.428571,42.0,0.462585,0,0,K-Means
1,G_2,19.428571,42.0,0.462585,0,0,K-Means
2,G_3,19.428571,42.0,0.462585,0,0,K-Means
3,G_4,19.428571,42.0,0.462585,0,0,K-Means
4,G_5,19.428571,42.0,0.462585,0,0,K-Means
5,G_6,19.428571,42.0,0.462585,0,0,K-Means
6,G_7,19.428571,42.0,0.462585,0,0,K-Means
7,G_8,19.428571,42.0,0.462585,0,0,K-Means
8,G_9,19.428571,42.0,0.462585,0,0,K-Means
9,G_10,19.428571,42.0,0.462585,0,0,K-Means


Cluster summary

In [32]:
if "cluster_str" in diversity_df.columns:
    cluster_summary = (
        diversity_df
        .groupby("cluster_str", dropna=False)
        .size()
        .rename("solutions")
        .reset_index()
    )

    display(cluster_summary)
else:
    print(
        "The diversity lens did not generate cluster labels. "
        "Check the selected dimensions and number of solutions."
    )

,cluster_str,solutions
0,0,25


Preference lens

In [33]:
preference_params = {
    "method": "TOPSIS",
    "maximize": ["satisfaction"],
    "minimize": ["effort"],
    "top_n": min(5, len(enriched_df)),
}

preference_lens = PreferenceLens()

preference_df = preference_lens.apply(
    enriched_df.copy(),
    preference_params,
)

print("Preference configuration:", preference_params)
print(f"Selected solutions: {len(preference_df)}")

Preference configuration: {'method': 'TOPSIS', 'maximize': ['satisfaction'], 'minimize': ['effort'], 'top_n': 5}
Selected solutions: 5


C:\Users\imagu\OneDrive\Escritorio\NRPExperiment_rename\src\analysis\lenses\preference.py:157: RuntimeWarning: invalid value encountered in divide
  scores = np.where(denom != 0, d_minus / denom, 0.0)


Results

In [34]:
preference_columns = [
    column
    for column in [
        "id",
        "satisfaction",
        "effort",
        "preference_score",
        "preference_rank",
        "preference_method",
    ]
    if column in preference_df.columns
]

preference_df[preference_columns]

,id,satisfaction,effort,preference_score,preference_rank,preference_method
0,G_1,19.428571,42.0,0.0,1,TOPSIS
1,G_2,19.428571,42.0,0.0,2,TOPSIS
2,G_3,19.428571,42.0,0.0,3,TOPSIS
3,G_4,19.428571,42.0,0.0,4,TOPSIS
4,G_5,19.428571,42.0,0.0,5,TOPSIS


## 7. Compare complementary Sets of Interest

For demonstration purposes, the diversity and preference outputs are
represented as two Sets of Interest through their solution identifiers.

A production analysis session can store the full subsets and their
provenance metadata. Here, the overlap is calculated explicitly to show
whether the two analytical rationales support the same alternatives.

In [35]:
def extract_solution_ids(df: pd.DataFrame) -> set:
    if df is None or df.empty or "id" not in df.columns:
        return set()

    return set(
        df["id"]
        .dropna()
        .astype(str)
        .tolist()
    )


if "cluster_str" in diversity_df.columns:
    cluster_counts = (
        diversity_df["cluster_str"]
        .dropna()
        .value_counts()
    )

    if not cluster_counts.empty:
        largest_cluster = cluster_counts.index[0]

        diversity_soi_df = diversity_df[
            diversity_df["cluster_str"] == largest_cluster
        ].copy()
    else:
        largest_cluster = None
        diversity_soi_df = diversity_df.copy()
else:
    largest_cluster = None
    diversity_soi_df = diversity_df.copy()


preference_soi_df = preference_df.copy()


diversity_soi_ids = extract_solution_ids(
    diversity_soi_df
)

preference_soi_ids = extract_solution_ids(
    preference_soi_df
)


overlap_ids = (
    diversity_soi_ids
    & preference_soi_ids
)

union_ids = (
    diversity_soi_ids
    | preference_soi_ids
)


jaccard_similarity = (
    len(overlap_ids) / len(union_ids)
    if union_ids
    else 0.0
)


soi_comparison = {
    "Diversity cluster": (
        str(largest_cluster)
        if largest_cluster is not None
        else "Not available"
    ),
    "Diversity SOI size": len(diversity_soi_ids),
    "Preference SOI size": len(preference_soi_ids),
    "Common solutions": len(overlap_ids),
    "Union size": len(union_ids),
    "Jaccard similarity": jaccard_similarity,
}


pd.Series(
    soi_comparison,
    name="SOI comparison",
).to_frame()

,SOI comparison
Diversity cluster,0
Diversity SOI size,25
Preference SOI size,5
Common solutions,5
Union size,25
Jaccard similarity,0.2


In [36]:
css_ids = sorted(overlap_ids)

css_df = enriched_df[
    enriched_df["id"].astype(str).isin(css_ids)
].copy()

print(f"CSS size: {len(css_df)}")
print("CSS solution IDs:", css_ids)

css_display_columns = [
    column
    for column in [
        "id",
        "satisfaction",
        "effort",
        *new_indicator_columns,
        "selected_ids",
        "selected_ids_str",
    ]
    if column in css_df.columns
]

css_df[css_display_columns]

CSS size: 5
CSS solution IDs: ['G_1', 'G_2', 'G_3', 'G_4', 'G_5']


,id,satisfaction,effort,productivity,selected_ids,selected_ids_str
0,G_1,19.428571,42.0,0.462585,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
1,G_2,19.428571,42.0,0.462585,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
2,G_3,19.428571,42.0,0.462585,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
3,G_4,19.428571,42.0,0.462585,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"
4,G_5,19.428571,42.0,0.462585,"[r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9]","r1, r10, r11, r12, r2, r3, r4_r5, r6, r8, r9"


In [38]:
if css_df.empty:
    support = {}

    for solution_id in diversity_soi_ids:
        support[solution_id] = (
            support.get(solution_id, 0) + 1
        )

    for solution_id in preference_soi_ids:
        support[solution_id] = (
            support.get(solution_id, 0) + 1
        )

    max_support = max(
        support.values(),
        default=0,
    )

    css_ids = sorted(
        solution_id
        for solution_id, count in support.items()
        if count == max_support
    )

    css_df = enriched_df[
        enriched_df["id"].astype(str).isin(css_ids)
    ].copy()

    print(
        "The direct intersection was empty. "
        "The CSS contains the alternatives with maximum support."
    )

    print("CSS solution IDs:", css_ids)

Export results

In [39]:
diversity_soi_df.to_csv(
    RESULTS_DIRECTORY
    / "diversity_soi.csv",
    index=False,
)

preference_soi_df.to_csv(
    RESULTS_DIRECTORY
    / "preference_soi.csv",
    index=False,
)

css_df.to_csv(
    RESULTS_DIRECTORY
    / "css.csv",
    index=False,
)

print("SOI and CSS results exported successfully.")

SOI and CSS results exported successfully.


## 8. Reproducibility checks

The final checks verify structural properties of the workflow rather than
machine-dependent execution time.

Execution time is reported but is not asserted because it depends on the
operating system, processor, Python environment, and background workload.

In [41]:
assert len(processed_models) >= 1
assert len(model.items) >= 1
assert len(pareto_front) >= 1
assert not pareto_df.empty
assert not enriched_df.empty
assert not diversity_df.empty
assert not preference_df.empty

assert set(OBJECTIVES).issubset(
    pareto_df.columns
)

assert all(
    solution.is_feasible
    for solution in pareto_front.solutions
)

assert not any(
    processed_model.relationships.get("couplings", [])
    for processed_model in processed_models
)

expected_output_files = [
    "greedy_pareto.csv",
    "execution_summary.csv",
    "diversity_soi.csv",
    "preference_soi.csv",
    "css.csv",
]

for file_name in expected_output_files:
    output_path = RESULTS_DIRECTORY / file_name

    assert output_path.exists(), (
        f"Expected output file was not generated: "
        f"{output_path}"
    )

print("All reproducibility checks passed.")

All reproducibility checks passed.


## 9. Relation to the research work

This notebook provides the programmatic counterpart of the interactive
workflow shown in the article.

The Streamlit interface and this notebook use the same underlying:

- NRP plugin;
- generic model representation;
- preprocessing engine;
- optimization problem representation;
- solver implementations;
- Pareto-front abstraction;
- enrichment functions;
- analytical lenses.

The notebook therefore demonstrates that the optimization and analytical
workflow can be executed independently of the graphical interface.

TEST

In [42]:
import subprocess


test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(test_result.stdout)

if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("The test suite failed.")

print("The complete test suite passed.")

........................................................................ [ 57%]
.....................................................                    [100%]
============================== warnings summary ===============================
tests/test_lens_diversity.py::test_diversity_lens_hdbscan_auto
tests/test_lens_diversity.py::test_diversity_lens_hdbscan_manual
tests/test_lens_diversity.py::test_diversity_lens_exclude_noise
  c:\Users\imagu\OneDrive\Escritorio\NRPExperiment_rename\venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
    warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
125 passed, 3 warnings in 8.73s

The complete test suite passed.
